In [87]:
import yfinance as yf
import pandas as pd
import os

In [65]:
# Daily USD/PHP from 2018 to today
fx = yf.download("PHP=X", start="2015-01-01", end="2026-05-04", 
                 interval="1d", auto_adjust=False, multi_level_index=False)


[*********************100%***********************]  1 of 1 completed


In [66]:
fx

,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2015-01-01,44.666000,44.666000,44.689999,44.667999,44.689999,0
2015-01-02,44.648998,44.648998,44.790001,43.814999,44.689999,0
2015-01-05,44.755001,44.755001,45.119999,44.830002,44.907001,0
2015-01-06,44.959999,44.959999,45.044998,44.777000,44.990002,0
2015-01-07,44.917999,44.917999,45.092999,44.874001,44.930000,0
...,...,...,...,...,...,...
2026-04-28,60.748001,60.748001,61.429001,60.106998,60.748001,0
2026-04-29,61.076000,61.076000,61.707001,61.064999,61.095001,0
2026-04-30,61.736000,61.736000,61.736000,61.106998,61.736000,0


In [67]:
fx.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2950 entries, 2015-01-01 to 2026-05-02
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  2950 non-null   float64
 1   Close      2950 non-null   float64
 2   High       2950 non-null   float64
 3   Low        2950 non-null   float64
 4   Open       2950 non-null   float64
 5   Volume     2950 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 161.3 KB


In [68]:
fx.isnull().sum()

Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64

In [69]:
fx.columns

Index(['Adj Close', 'Close', 'High', 'Low', 'Open', 'Volume'], dtype='object')

In [70]:
usd_php = fx[["Close"]].rename(columns={"Close": "usd_php"})

In [71]:
usd_php = usd_php.reset_index() # date is alr in datetime
usd_php.columns = ["date", "usd_php"]
usd_php["date"] = pd.to_datetime(usd_php["date"])

In [72]:
usd_php.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2950 entries, 0 to 2949
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   date     2950 non-null   datetime64[ns]
 1   usd_php  2950 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 46.2 KB


In [78]:
usd_php.columns
usd_php.index

RangeIndex(start=0, stop=2950, step=1)

In [79]:
usd_php = usd_php.set_index("date").sort_index()

In [81]:
usd_php.loc["2026-04-01":, :]# proves that date is dt alr

,usd_php
date,
2026-04-01,60.574001
2026-04-02,60.169998
2026-04-03,60.308998
2026-04-06,60.215000
2026-04-07,59.978001
2026-04-08,59.849998
2026-04-09,59.521999
2026-04-10,59.648998
2026-04-13,59.928001


In [83]:
# aggregate to weekly (Friday close, ISO Monday-anchored week)
weekly = usd_php.resample("W-FRI").last()  # Friday close
weekly["usd_php_mean"] = usd_php.resample("W-FRI").mean()
weekly["usd_php_high"] = usd_php.resample("W-FRI").max()
weekly["usd_php_low"] = usd_php.resample("W-FRI").min()

# anchor to Monday-of-the-week (matches DOE alignment)
weekly.index = weekly.index - pd.Timedelta(days=4)
weekly.index.name = "week_start"

In [84]:
weekly

,usd_php,usd_php_mean,usd_php_high,usd_php_low
week_start,,,,
2014-12-29,44.648998,44.657499,44.666000,44.648998
2015-01-05,44.743999,44.872600,44.986000,44.743999
2015-01-12,44.521000,44.616000,44.755001,44.507999
2015-01-19,44.085999,44.338999,44.507000,44.085999
2015-01-26,43.888000,43.991001,44.075001,43.888000
...,...,...,...,...
2026-04-06,59.648998,59.842799,60.215000,59.521999
2026-04-13,59.988998,59.952801,60.096001,59.705002
2026-04-20,60.445999,60.025600,60.445999,59.554001


In [90]:
os.makedirs("data/semi-cleaned", exist_ok=True)
weekly.to_parquet("data/semi-cleaned/fx_usd_php.parquet")